In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import periodogram
from ordpy import permutation_entropy
from vmdpy import VMD
# from statsforecast import StatsForecast
# from statsforecast.utils import ConformalIntervals
# from statsforecast.models import SeasonalExponentialSmoothing, ARIMA, ADIDA, GARCH, MSTL, MFLES, TBATS

from var import DATA_OUT, START_DATE
from scintill_ai.preprocess import get_time_filtering_and_features

In [ ]:
df = pd.read_parquet(Path(DATA_OUT, 'df.parquet'), engine='pyarrow').drop(
    columns=['s4_max']
)

df = get_time_filtering_and_features(
    df, hour_start=20, hour_stop=6, ema_cols={'s4_mean': [5]}, lag_cols={'h_tmk': [180]},
)

# Forecastability Metrics

## Permutation Entropy (PE)

PE evaluates the diversity of ordinal patterns in a time series, measuring complexity based on these. It is defined as
$$H_\mathrm{P} = - \frac{1}{\log_2 m!} \sum_{i=1}^{m!} p_i \log_2 p_i,$$ and it looks at the relative order of consecutive values, measuring the diversity of such ordinal patterns

**Interpretation**
- $H_\mathrm{P} \sim 0$: repeated ordinal patterns $\rightarrow$ more deterministic signal $\rightarrow$ higher forecastability
- $H_\mathrm{P} \sim 1$: all patterns equally likely $\rightarrow$ more random signal $\rightarrow$ lower forecastability

**Estimated $H_\mathrm{P}$ is $0.95$**

In [ ]:
permutation_entropy(
    df['s4_mean'].to_numpy(),
    dx=5,
    taux=1,
).round(3)

## Forecastable Component Analysis (ForeCA)

ForeCA defines a forecastability metric based on spectral entropy,
$$\Omega(y) = 1 - \frac{H_\mathrm{spec}(y)}{\log_2 N},$$

where:
- $H_\mathrm{spec}(y)$ is the spectral entropy of the signal $y$, computed from the normalized power spectral density
- $N$ are the frequency bins used in the spectral estimate (typically equal to the FFT resolution)
- $\log_2 N$ is the maximum possible entropy for a flat (uniform) spectrum

$\Omega(y)$ is normalised to lie in $[0,1]$ and formalises the notion that predictable signals have concentrated spectral energy. Thus, it provides an information-theoretic baseline for how much structure is present in the signal that a forecasting model could potentially exploit.

**Interpretation**
- $\Omega = 0$: corresponds to white noise, with energy evenly spread across all frequencies
- $\Omega = 1$: corresponds to a perfectly predictable signal with all spectral energy at a single frequency

**Estimated $\Omega$ is $0.35$**

In [ ]:
def foreca(s):
    freqs, psd = periodogram(s)
    psd = psd / np.sum(psd)
    entropy = - np.sum(psd * np.log2(psd + 1e-12))
    max_entropy = np.log2(len(psd))
    return 1 - (entropy / max_entropy)

In [ ]:
print(f"{foreca(df['s4_mean'].dropna()):.2f}")

## Maximum Lyapunov Exponent (MLE)

Lyapunov exponents quantify how long a chaotic or nonlinear system remains predictable, measuring how quickly nearby state space trajectories diverge over time. If two initial points are infinitesimally close, their distance $d(t)$ after time $t$ evolves as $$d(t) \sim d(0)e^{\lambda_{\mathrm{max}}t},$$ where $\lambda_{\mathrm{max}}$ is the largest Lyapunov exponent, representing the average exponential divergence rate of nearby trajectories.

If $\lambda_{\mathrm{max}} \gt 0$, the system is sensitive to initial conditions and exhibits **chaos**. The inverse of this exponent, called the **Lyapunov time** $\tau$, represents the characteristic timescale on which the system is chaotic, thus approximating a **theoretical forecasting limit** (in time steps), given by $$\tau \sim \lambda_{\mathrm{max}}^{-1}.$$
By contrast, if $\lambda_{\mathrm{max}} \leq 0$, the system does not exhibit exponential divergence and may be reliably predictable (*e.g.*, fixed-point or periodic dynamics).

**Estimated MLE**
- 25.3k points between 2023-02-01 and 2023-03-15 yielded $\lambda_{\mathrm{max}} \sim 0.0739$ $\longrightarrow$ $\tau \sim 13$ (minutes)

In [ ]:
from sklearn.neighbors import NearestNeighbors
from scipy.stats import linregress

def time_delay_embedding(series, dim=3, delay=1):
    N = len(series) - (dim - 1) * delay
    if N <= 0:
        raise ValueError("Serie troppo corta per embedding richiesto")
    return np.array([series[i:i + delay * dim:delay] for i in range(N)])

def lyapunov_exponent(series, dim=3, delay=1, max_t=20, eps=1e-4, min_neighbors=10):
    embedded = time_delay_embedding(series, dim=dim, delay=delay)
    N = len(embedded)
    l_exp_values = []

    for i in range(N - max_t):
        x_i = embedded[i].reshape(1, -1)

        # Trova vicini (escludendo vicini temporali troppo prossimi)
        nbrs = NearestNeighbors(n_neighbors=min_neighbors + 1).fit(embedded)
        distances, indices = nbrs.kneighbors(x_i)

        for j in indices[0][1:]:  # salta il primo (sé stesso)
            if abs(i - j) < delay * dim:
                continue

            dists = []
            for k in range(max_t):
                if i + k >= N or j + k >= N:
                    break
                dist = np.linalg.norm(embedded[i + k] - embedded[j + k])
                if dist < eps:
                    continue
                dists.append(np.log(dist))

            if len(dists) > 2:
                slope, _, _, _, _ = linregress(range(len(dists)), dists)
                l_exp_values.append(slope)

    if len(l_exp_values) == 0:
        return np.nan

    return np.mean(l_exp_values)

In [ ]:
# 25.329 points
lyapunov_exponent(
    df.loc['2023-02-01': '2023-03-15', 's4_mean'].dropna(),
)

## Variational Mode Decomposition (VMD)

[sktime](https://github.com/sktime/sktime/tree/main/sktime/libs/vmdpy), [paper](https://ww3.math.ucla.edu/camreport/cam14-16.pdf)

$$x(t) = \sum_{k=1}^{K} u_k (t)$$
- $x(t)$ is the original signal
- $K$ are the desidered modes
- $u_k (t)$ is the $k$-th mode deriving from the decomposition

In [ ]:
modes = 3

u, u_hat, omega = VMD(
    df.loc['2023-03-01': '2023-03-15', 's4_mean'].dropna().to_numpy(),
    alpha=2_000,
    tau=0.,
    K=modes,
    DC=0,
    init=1,
    tol=1e-6,
)

In [ ]:
xlims = (0, u.shape[1])
ylims = [
    (0, 0.60),     # Mode 1
    (-0.45, 0.45), # Mode 2
    (-0.45, 0.45), # Mode 3
]

fig, axes = plt.subplots(modes, 1, figsize=(12, 7), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(u[i], color=f"C{i}")
    ax.set_ylabel(f"Mode {i+1}")
    ax.grid(True)
    ax.set_ylim(ylims[i])
    ax.set_xlim(xlims)
axes[-1].set_xlabel("")

plt.subplots_adjust(hspace=0.1)
plt.tight_layout()
plt.show()

$$f_\mathrm{Hz} = \frac{\omega_{\mathrm{rad/sample}}}{2\pi} f_\mathrm{samp}$$

In [ ]:
fs = 1 / 60
freqs_hz = omega[-1] * fs / (2 * np.pi)

for i, f in enumerate(freqs_hz):
    print(f"Mode {i+1}\t Central frequency ≈ {f:.1e} Hz\t ~ {f*60*60*10:.1f} cycles/10h-window")

In [ ]:
for i in range(u.shape[0]):
    pe_ = permutation_entropy(
        u[i],
        dx=5,
        taux=1,
    )
    fca_ = foreca(u[i])
    print(f"Mode {i+1}\t PE: {pe_:.2f}\t ForeCA: {fca_:.2f}")

In [ ]:
magnitudes = np.abs(u_hat)
plt.plot(magnitudes)
plt.show()

In [ ]:
# plt.figure(figsize=(10, 6))
# for i in range(modes):
#     plt.plot(u[i], label=f"Mode {i+1}")
# plt.title("VMD Decomposition: Structured Modes")
# plt.tight_layout()
# plt.grid(True)
# plt.show()

## (Auto)Correlations

In [ ]:
# pd.plotting.autocorrelation_plot(df.loc['2024-01-03':'2024-01-07','s4_mean'])

In [ ]:
# df['s4_mean'].diff(24 * 60)

pd.plotting.autocorrelation_plot(
    df['s4_mean'].diff(60 * 24).loc['2024-01-03':'2024-01-07']
)

In [ ]:
from statsmodels.tsa.stattools import adfuller

In [ ]:
adfuller(
    df.loc['2024-05-10':'2024-05-15','s4_mean'].fillna(0)
)

In [ ]:
# non_nan = df['s4_mean'].notna()
# seg_leng, seg_start, seg_end = [], [], []

# for g_id, g_ in non_nan.groupby(
#     (non_nan != non_nan.shift()).cumsum()
# ):
#     if g_.all():
#         seg_leng.append(len(g_))
#         seg_start.append(g_.index[0])
#         seg_end.append(g_.index[-1])

# df_non_nans = pd.DataFrame(
#     {
#         'dt_start': seg_start,
#         'dt_end': seg_end,
#         'length_mins': seg_leng,
#         'length_days': [round(l_/(60*24),1) for l_ in seg_leng],
#     }
# ).set_index('dt_start')

# df_non_nans['length_days_train'] = round(0.8 * df_non_nans['length_days'], 1)
# df_non_nans['length_days_test'] = round(0.2 * df_non_nans['length_days'], 1)

In [ ]:
# # 1 October to mid-April is when you expect the most scintillation
# df_non_nans.sort_values(ascending=False, by='length_mins').head(15)

In [ ]:
TRAIN_START, TRAIN_STOP = '2022-10-01', '2023-03-17'

In [ ]:
df.columns.to_list()

In [ ]:
# px.scatter(x=df.loc[TRAIN_START:TRAIN_STOP, 's4_mean'].values, y=df.loc[TRAIN_START:TRAIN_STOP, 'h_tmk'].shift(3*60).values)

In [ ]:
px.scatter(x=df.loc[TRAIN_START:TRAIN_STOP, 's4_mean'].values, y=df.loc[TRAIN_START:TRAIN_STOP, 'h_tmk'].shift(180).values)

In [ ]:
# df.loc[df['s4_mean'].ge(0.4), 'f10.7_adj'].plot.kde()

In [ ]:
import plotly.graph_objects as go

lags = range(-60*6, 60*6+1, 15)
fig = go.Figure()

for col in df.columns.to_list():
    corr_values = [df.loc[TRAIN_START:TRAIN_STOP, 's4_mean'].shift(lag).corr(df.loc[TRAIN_START:TRAIN_STOP, col]) for lag in lags]
    fig.add_trace(go.Scatter(x=list(lags), y=corr_values, mode='lines', name=col))

fig.update_layout(
    xaxis_title="Lag (min)",
    width=1000,
    height=500,
    margin=dict(l=10, r=10, t=20, b=10)
)
fig.show()

# Nixtla StatsForecast

In [ ]:
# df_plt = df.loc['2023', 's4_mean']

# fig, ax = plt.subplots(figsize=(16, 5))

# ax.plot(df_plt)
# ax.grid(True, linestyle='-', linewidth=0.4, alpha=0.5)
# ax.set_xlim(df_plt.index[0], df_plt.index[-1])
# ax.set_ylim(df_plt.min())

# plt.show()

In [ ]:
X_cols = ['h_tmk']
y_col = ['s4_mean']

In [ ]:
# df_train = df.loc['2024-02-01':'2024-02-15', y_col].copy().reset_index(names='ds').rename(columns={'s4_mean': 'y'})
# df_train["unique_id"] = "s4_forecast"

# df_test = df.loc['2024-02-18 19:30':'2024-02-20', y_col].copy().reset_index(names='ds').rename(columns={'s4_mean': 'y'})
# df_test["unique_id"] = "s4_forecast"

In [ ]:
df_train = df.loc[
    '2024-01-01':'2024-03-31', y_col
].copy().reset_index(names='ds').rename(
    columns={'s4_mean': 'y'}
).fillna(0)
df_train["unique_id"] = "s4_forecast"

df_test = df.loc[
    '2024-04-01':'2024-04-10', y_col
].copy().reset_index(names='ds').rename(
    columns={'s4_mean': 'y'}
)
df_test["unique_id"] = "s4_forecast"

In [ ]:
horizon = 15

intervals = ConformalIntervals(h=horizon, n_windows=2)

models = [
    # SeasonalExponentialSmoothing(season_length=10*24*60, alpha=0.1, prediction_intervals=intervals),
    # AutoARIMA(
    #     seasonal=True,
    #     season_length=24*60,
    #     stepwise=True,
    #     approximation=True,
    #     d=1,
    #     D=1,
    # )
    # ADIDA(),
    # ARIMA(
    #     order=(2,1,2),
    #     seasonal_order=(1,0,1),
    #     include_drift=True,
    #     prediction_intervals=intervals
    # ),
    MSTL(season_length=24*60)
]

sf = StatsForecast(
    models=models, 
    freq='1min',
    n_jobs=-1,
)

In [ ]:
sf.fit(df=df_train, prediction_intervals=intervals)

In [ ]:
df_pred = sf.predict(
    h=horizon,
    X_df=df_test.drop(columns=['y']).head(horizon),
    level=[90, 95],
)
df_pred['y'] = df_test['y'].head(horizon)

In [ ]:
model = 'MSTL'
cl = 95

plt.figure(figsize=(15, 5))
plt.plot(df_pred["ds"], df_pred[f"{model}"], label=f"{model}", c="tab:blue", ls='-', lw=0.8, marker='o')
plt.fill_between(df_pred["ds"], df_pred[f"{model}-lo-{cl}"], df_pred[f"{model}-hi-{cl}"], color="tab:blue", alpha=0.2)
plt.plot(df_pred["ds"], df_pred["y"], label="Real", c="k", ls="-", lw=0.8, marker="o")

plt.legend()
plt.grid()
plt.show()

# Random forest + [EnbPI](https://colab.research.google.com/drive/1i9zpgOppIjNVN4K7xaFcFmlCiuFn5Pez) (o [ACI](https://mapie.readthedocs.io/en/latest/examples_regression/4-tutorials/plot_ts-tutorial.html))

In [ ]:
from typing import cast

import numpy as np
import pandas as pd
from matplotlib import pylab as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from mapie._typing import NDArray
from mapie.metrics import regression_coverage_score, coverage_width_based, regression_mean_width_score
from mapie.regression import MapieTimeSeriesRegressor
from mapie.subsample import BlockBootstrap
import shap

In [ ]:
ALPHAS = [1 - 0.80, 1 - 0.90, 1 - 0.95]
GAP = 1

In [ ]:
df['s4_lag_10mins'] = df['s4_mean'].shift(10)

In [ ]:
X_cols = [
    'h_tmk',
    'f10.7_adj',
    'sza',
    's4_lag_10mins',
]

y_col = ['s4_mean']

X_train, X_test = df.loc['2024-01-01':'2024-03-31', X_cols].copy(), df.loc['2024-04-01':'2024-04-10', X_cols].copy()
y_train, y_test = df.loc['2024-01-01':'2024-03-31', y_col].copy().fillna(0), df.loc['2024-04-01':'2024-04-10', y_col].copy().fillna(0)

In [ ]:
# n_iter = 50
# n_splits = 5
# tscv = TimeSeriesSplit(n_splits=n_splits)
# random_state = 42
# rf_model = RandomForestRegressor(random_state=random_state)
# rf_params = {
#     "max_depth": [int(x) for x in np.linspace(2, 10, num=5)],
#     "n_estimators": [int(x) for x in np.linspace(10, 100, num=10)],
# }
# cv_obj = RandomizedSearchCV(
#     rf_model,
#     param_distributions=rf_params,
#     n_iter=n_iter,
#     cv=tscv,
#     scoring="neg_root_mean_squared_error",
#     random_state=random_state,
#     verbose=0,
#     n_jobs=-1,
# )
# cv_obj.fit(X_train, y_train.values)
# model = cv_obj.best_estimator_

In [ ]:
# model

In [ ]:
model = RandomForestRegressor(
    # max_depth=4, n_estimators=30, random_state=42
    max_depth=4, n_estimators=20, random_state=42
)

In [ ]:
cv_mapietimeseries = BlockBootstrap(
    n_resamplings=10, n_blocks=10, overlapping=False, random_state=42,
)

In [ ]:
results_no_pfit = []
for alpha in ALPHAS:
    enbpi_no_pfit = MapieTimeSeriesRegressor(
        model,
        method='enbpi',
        cv=cv_mapietimeseries,
        agg_function='mean',
        n_jobs=-1,
    )

    enbpi_no_pfit.fit(X_train, y_train.values)
    y_pred_no_pfit, y_pis_no_pfit = enbpi_no_pfit.predict(
        X_test, alpha=alpha, ensemble=True,
    )

    results_no_pfit.append(
        {
            'y_pred': y_pred_no_pfit,
            'y_pis': y_pis_no_pfit,
            'coverage': regression_coverage_score(y_test, y_pis_no_pfit[:, 0, 0], y_pis_no_pfit[:, 1, 0]),
            'width': regression_mean_width_score(y_pis_no_pfit[:, 1, 0], y_pis_no_pfit[:, 0, 0]),
            'alpha': alpha,
        }
    )

results_pfit = []
for alpha in ALPHAS:
    enbpi_pfit = MapieTimeSeriesRegressor(
        model,
        method="enbpi",
        cv=cv_mapietimeseries,
        agg_function="mean",
        n_jobs=-1,
    )

    enbpi_pfit.fit(X_train, y_train)

    y_pred_pfit = np.zeros(y_test.shape)
    y_pis_pfit = np.zeros((len(y_test), 2, 1))
    # Inizializza le prime predizioni
    y_pred_pfit[:GAP], y_pis_pfit[:GAP, :, :] = enbpi_pfit.predict(
        X_test.iloc[:GAP, :], alpha=alpha, ensemble=True,
    )

    for step in range(GAP, len(X_test), GAP):
        enbpi_pfit.partial_fit(
            X_test.iloc[(step - GAP):step, :],
            y_test.iloc[(step - GAP):step],
        )
        y_pred_pfit[step:step + GAP], y_pis_pfit[step:step + GAP, :, :] = enbpi_pfit.predict(
            X_test.iloc[step:(step + GAP), :],
            alpha=alpha,
            ensemble=True,
        )

    results_pfit.append({
        "y_pred": y_pred_pfit,
        "y_pis": y_pis_pfit,
        "coverage": regression_coverage_score(y_test, y_pis_pfit[:, 0, 0], y_pis_pfit[:, 1, 0]),
        "width": regression_mean_width_score(y_pis_pfit[:, 1, 0], y_pis_pfit[:, 0, 0]),
        "alpha": alpha
    })

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.set_ylabel("<S4>", fontsize=16)
ax.plot(y_train, lw=1, label="Actual (train)", c="C1")
plt.xticks(rotation=45)
ax.legend(prop={'size': 10}, loc='upper right', frameon=False)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.set_ylabel("<S4>", fontsize=16)
ax.plot(y_test, lw=1, label="Actual (test)", c="C1")
ax.plot(
    y_test.index,
    results_no_pfit[0]["y_pred"],
    lw=1,
    c="C2",
    label="Forecast"
)

for result_ in results_no_pfit:
    y_pis = cast(NDArray, result_["y_pis"])
    ax.fill_between(
        y_test.index,
        y_pis[:, 0, 0],
        y_pis[:, 1, 0],
        alpha=0.2,
        label=f"{1 - result_['alpha']:.0%} CL (cover. {result_['coverage']:.0%} – mean width {result_['width']:.2f})",
    )

ax.set_title('EnbPI, without partial_fit', fontweight="bold", size=18)
plt.xticks(rotation=45)
ax.legend(prop={'size': 10}, loc='upper right', frameon=False)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.set_ylabel("<S4>", fontsize=16)
ax.plot(y_test, lw=1, label="Actual (test)", c="C1")
ax.plot(
    y_test.index,
    results_pfit[0]["y_pred"],
    lw=1,
    c="C2",
    label="Forecast"
)

for result_ in results_pfit:
    y_pis = cast(NDArray, result_["y_pis"])
    ax.fill_between(
        y_test.index,
        y_pis[:, 0, 0],
        y_pis[:, 1, 0],
        alpha=0.2,
        label=f"{1 - result_['alpha']:.0%} CL (cover. {result_['coverage']:.0%} – mean width {result_['width']:.2f})",
    )

ax.set_title('EnbPI, with partial_fit', fontweight="bold", size=18)
plt.xticks(rotation=45)
ax.legend(prop={'size': 10}, loc='upper right', frameon=False)

plt.show()

In [ ]:
def rrmse(y_true, y_pred, digit=3):
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
    mean_true = np.mean(y_true)
    return np.round(rmse / mean_true, digit)

def rmse(y_true, y_pred, digit=3):
    return np.round(np.sqrt(np.mean((y_pred - y_true) ** 2)), digit)

In [ ]:
rmse(y_true=y_test.values[:,0], y_pred=results_no_pfit[0]['y_pred'])

In [ ]:
rrmse(y_true=y_test.values[:,0], y_pred=results_no_pfit[0]['y_pred'])

In [ ]:
model.fit(X_train, y_train.values)
explainer = shap.Explainer(model, X_train)

In [ ]:
shap.plots.beeswarm(
    explainer(X_train, check_additivity=False)
)

In [ ]:
X_train, X_test = df.loc['2022-09-06 07:00':'2022-10-08', X_cols].copy(), df.loc['2022-10-08':'2022-10-15', X_cols].copy()
y_train, y_test = df.loc['2022-09-06 07:00':'2022-10-08', y_col].copy().fillna(0), df.loc['2022-10-08':'2022-10-15', y_col].copy().fillna(0)

In [ ]:
n_iter = 50
n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)
random_state = 42
rf_model = RandomForestRegressor(random_state=random_state)
rf_params = {
    "max_depth": [int(x) for x in np.linspace(2, 10, num=5)],
    "n_estimators": [int(x) for x in np.linspace(10, 100, num=10)],
}
cv_obj = RandomizedSearchCV(
    rf_model,
    param_distributions=rf_params,
    n_iter=n_iter,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    random_state=random_state,
    verbose=0,
    n_jobs=-1,
)
cv_obj.fit(X_train, y_train.values)
model = cv_obj.best_estimator_

In [ ]:
model

In [ ]:
model = RandomForestRegressor(
    max_depth=6, n_estimators=60, random_state=42
)

In [ ]:
cv_mapietimeseries = BlockBootstrap(
    n_resamplings=10, n_blocks=10, overlapping=False, random_state=42,
)

In [ ]:
results_no_pfit = []
for alpha in ALPHAS:
    enbpi_no_pfit = MapieTimeSeriesRegressor(
        model,
        method='enbpi',
        cv=cv_mapietimeseries,
        agg_function='mean',
        n_jobs=-1,
    )

    enbpi_no_pfit.fit(X_train, y_train.values)
    y_pred_no_pfit, y_pis_no_pfit = enbpi_no_pfit.predict(
        X_test, alpha=alpha, ensemble=True,
    )

    results_no_pfit.append(
        {
            'y_pred': y_pred_no_pfit,
            'y_pis': y_pis_no_pfit,
            'coverage': regression_coverage_score(y_test, y_pis_no_pfit[:, 0, 0], y_pis_no_pfit[:, 1, 0]),
            'width': regression_mean_width_score(y_pis_no_pfit[:, 1, 0], y_pis_no_pfit[:, 0, 0]),
            'alpha': alpha,
        }
    )

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.set_ylabel("<S4>", fontsize=16)
ax.plot(y_train, lw=1, label="Actual (train)", c="C1")
plt.xticks(rotation=45)
ax.legend(prop={'size': 10}, loc='upper right', frameon=False)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.set_ylabel("<S4>", fontsize=16)
ax.plot(y_test, lw=2, label="Actual (test)", c="C1")
ax.plot(
    y_test.index,
    results_no_pfit[0]["y_pred"],
    lw=2,
    c="C2",
    label="Forecast"
)

for result_ in results_no_pfit:
    y_pis = cast(NDArray, result_["y_pis"])
    ax.fill_between(
        y_test.index,
        y_pis[:, 0, 0],
        y_pis[:, 1, 0],
        alpha=0.2,
        label=f"{1 - result_['alpha']:.0%} CL (cover. {result_['coverage']:.0%} – mean width {result_['width']:.2f})",
    )

ax.set_title('EnbPI, without partial_fit', fontweight="bold", size=18)
plt.xticks(rotation=45)
ax.legend(prop={'size': 10}, loc='upper right', frameon=False)

plt.show()

In [ ]:
rmse(y_true=y_test.values[:,0], y_pred=results_no_pfit[0]['y_pred'])

In [ ]:
rrmse(y_true=y_test.values[:,0], y_pred=results_no_pfit[0]['y_pred'])

In [ ]:
model.fit(X_train, y_train.values)
explainer = shap.Explainer(model, X_train)

In [ ]:
shap.plots.beeswarm(
    explainer(X_train, check_additivity=False)
)